# 07 导航与基础本体控制

## 学习目标

- 语义导航到区域。
- 控制头部和升降高度。
- 谨慎理解底盘相对运动。
- 每个动作后检查 `MissionStatus`。

参考文献：文档第 6.2、6.5 章。

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import DEFAULT_WS_URL

WS_URL = DEFAULT_WS_URL
print("WS_URL =", WS_URL)

In [ ]:
CONNECT_ROBOT = False
ENABLE_NAVIGATION = False
ENABLE_HEAD = False
ENABLE_HEIGHT = False
ENABLE_MOVE = False

AREA_NAME = "客厅"
HEAD_RAD = 0.83
HEIGHT_M = 0.22
MOVE_DISTANCE_M = 0.05
MOVE_ANGLE_RAD = 0.0

In [ ]:
from helpers import connect_robot, disconnect_safely, print_mission_status, print_safety_gate, require_enabled
from bajie_sdk import SemanticNavigationRequest

robot = connect_robot(WS_URL) if CONNECT_ROBOT else None
print_safety_gate(connected=robot is not None, enabled=ENABLE_NAVIGATION, action="semantic navigation")

In [ ]:
if robot and ENABLE_HEAD:
    st = robot.eco_setRobotHead(HEAD_RAD, timeout_sec=30.0)
    print_mission_status(st)
else:
    print("Head control disabled.")

In [ ]:
if robot and ENABLE_HEIGHT:
    st = robot.eco_setRobotHeight(HEIGHT_M, timeout_sec=30.0)
    print_mission_status(st)
else:
    print("Height control disabled.")

In [ ]:
if robot and ENABLE_NAVIGATION:
    req = SemanticNavigationRequest(area_id="", area_name=AREA_NAME)
    st = robot.eco_navigateToSemanticArea(req, timeout_sec=120.0)
    print_mission_status(st)
else:
    print("Navigation disabled.")

In [ ]:
if robot and ENABLE_MOVE:
    require_enabled(ENABLE_MOVE, "eco_moveChassis")
    st = robot.eco_moveChassis(MOVE_DISTANCE_M, MOVE_ANGLE_RAD, timeout_sec=30.0)
    print_mission_status(st)
else:
    print("Chassis move disabled by default.")

In [ ]:
disconnect_safely(robot)

## 机器人背景知识

- 导航依赖地图和语义区域。
- 头部角度单位是弧度。
- 升降高度有范围限制，SDK 会对越界值截断并告警。
- 底盘移动是真实运动，运行前要确认周围安全。

## 故障排查卡片

- 区域名错误：先用 `eco_robotFurniture()` 看语义地图信息。
- 被障碍物挡住：清理路径或换目标点。
- 任务失败：先看 `error_info.code` 和 `error_info.message`。

## 小练习

查询家具列表，找出一个真实区域名替换 `AREA_NAME`。